# Map Arvoredo dataset using Follium

In [1]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt
import json
from folium import GeoJson
import folium

In [2]:
# load the CSV file into a DataFrame
csv_path = "../data/arvoredo_cleaned.csv"
df = pd.read_csv(csv_path)
df.head()

,Código SIG Novo,Morada,Espécie,PAP,Manutenção,Ocupação,Local,Tipologia,Freguesia,Nome Vulgar,longitude,latitude
0,1,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095224,38.774517
1,2,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095219,38.774579
2,3,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095178,38.774387
3,4,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095173,38.774452
4,5,Alameda dos Oceanos,Quercus robur,NaN,JF,Árvore,Via pública,Caldeira,Parque das Nações,carvalho-roble;carvalho-alvarinho,-9.095168,38.774515


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 78201 entries, 0 to 78200
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Código SIG Novo  78201 non-null  int64  
 1   Morada           78027 non-null  str    
 2   Espécie          78201 non-null  str    
 3   PAP              63285 non-null  str    
 4   Manutenção       72795 non-null  str    
 5   Ocupação         73796 non-null  str    
 6   Local            75066 non-null  str    
 7   Tipologia        65736 non-null  str    
 8   Freguesia        78200 non-null  str    
 9   Nome Vulgar      78023 non-null  str    
 10  longitude        78201 non-null  float64
 11  latitude         78201 non-null  float64
dtypes: float64(2), int64(1), str(9)
memory usage: 14.4 MB


In [4]:
# count NaN values in each column
nan_counts = df.isna().sum()
nan_counts

Código SIG Novo        0
Morada               174
Espécie                0
PAP                14916
Manutenção          5406
Ocupação            4405
Local               3135
Tipologia          12465
Freguesia              1
Nome Vulgar          178
longitude              0
latitude               0
dtype: int64

In [5]:
df = df.fillna("Nao indentificada")
counts = df.isna().sum()
counts

Código SIG Novo    0
Morada             0
Espécie            0
PAP                0
Manutenção         0
Ocupação           0
Local              0
Tipologia          0
Freguesia          0
Nome Vulgar        0
longitude          0
latitude           0
dtype: int64

In [10]:
na_counts = {
    col: df[col].isin(["Nao indentificada", "Não identificada"]).sum()
    for col in ["Espécie", "Nome Vulgar"]
}
na_counts

{'Espécie': np.int64(8541), 'Nome Vulgar': np.int64(8719)}

In [9]:
freguesia_counts = df['Freguesia'].value_counts()
freguesia_counts

Freguesia
Parque das Nações          8551
Lumiar                     6723
Avenidas Novas             6034
Benfica                    5783
Alvalade                   4843
Marvila                    4792
Belém                      4210
Estrela                    3617
Carnide                    3286
São Domingos de Benfica    3182
Olivais                    2902
Arroios                    2875
Santa Clara                2808
Penha de França            2509
Alcântara                  2421
Santo António              2338
Areeiro                    2241
Ajuda                      1936
Campolide                  1872
Santa Maria Maior          1335
Campo de Ourique           1282
Misericórdia               1042
São Vicente                 924
Beato                       694
Nao indentificada             1
Name: count, dtype: int64

In [6]:
# optimize data
# keep only essential columns
df_minimal = df[['latitude', 'longitude', 'Nome Vulgar', 'Espécie', 'Local']].copy()

# round coordinates to 4 decimals (still city-accurate)
df_minimal['latitude'] = df_minimal['latitude'].round(4)
df_minimal['longitude'] = df_minimal['longitude'].round(4)

In [7]:
# create a map centered on the average coordinates of the trees
from folium.plugins import FastMarkerCluster

map_center = [df_minimal["latitude"].mean(), df_minimal["longitude"].mean()]
m = folium.Map(location=map_center, zoom_start=12, tiles="OpenStreetMap")

coords = df_minimal[["latitude", "longitude", "Espécie", "Nome Vulgar"]].values.tolist()

# change icon to leaf and add species name to popup
callback = """
function (row) {

    var icon = L.AwesomeMarkers.icon({
        icon: 'leaf',
        prefix: 'fa',
        markerColor: 'green'
    });

    var marker = L.marker(
        [row[0], row[1]],
        {icon: icon}
    );

    marker.bindPopup(
        '<b>Especie:</b> ' + row[2]
        '<br><b>Nome Vulgar:</b> ' + row[3]
    );

    return marker;
}
"""
# change icon colour
icon_create_function = """
function(cluster) {
    var childCount = cluster.getChildCount();
    var c = ' marker-cluster-large';

    if (childCount < 5) {
        c = ' marker-cluster-small';
    } else if (childCount < 20) {
        c = ' marker-cluster-medium';
    }

    return new L.DivIcon({
        html: '<div><span>' + childCount + '</span></div>',
        className: 'marker-cluster' + c,
        iconSize: new L.Point(40, 40)
    });
}
"""

# add CSS for colours
css = """
<style>
.marker-cluster-small {
    background-color: rgba(232,245,233,0.6);
}
.marker-cluster-small div {
    background-color: rgba(232,245,233,0.9);
    color: #1b5e20;
}

.marker-cluster-medium {
    background-color: rgba(200,230,201,0.6);
}
.marker-cluster-medium div {
    background-color: rgba(200,230,201,0.9);
    color: #1b5e20;
}

.marker-cluster-large {
    background-color: rgba(102,187,106,0.6);
}
.marker-cluster-large div {
    background-color: rgba(27,94,32,0.9);
    color: white;
}
</style>
"""

# add the CSS to the map
m.get_root().html.add_child(folium.Element(css))


# create cluster map
fast_cluster = FastMarkerCluster(
    coords,
    name="Lisbon Trees",
    callback=callback,
    icon_create_function=icon_create_function,
)

# add the FastMarkerCluster to the map
fast_cluster.add_to(m)

# additional basemaps
folium.TileLayer(
    "CartoDB Positron",
    name="Light Map"
).add_to(m)

folium.TileLayer(
    "CartoDB Voyager",
    name="Voyager"
).add_to(m)

folium.TileLayer(
    "Esri.WorldImagery",
    name="Satellite"
).add_to(m)

folium.LayerControl().add_to(m)

In [8]:
# Save the map to an HTML file
m.save('../data/arvoredo_map_layers.html')
print("Map saved to '../data/arvoredo_map_layers.html'")

Map saved to '../data/arvoredo_map_layers.html'
